In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import random

# --- 1. SYSTEM PARAMETERS (From the Hardware/Paper config) ---
Ts = 0.1       # Simulation step size (s)
mass = 0.60    # Mass of ego vehicle (kg)
HWT = 0.70     # Headway Time (s)
dss = 0.25     # Standstill distance (m)
v_set = 0.38   # Driver set cruise speed (m/s)
a_min, a_max = -0.60, 0.40 # Actuator limits
kT = 0.520     # Motor torque constant

# MPC Tuning Weights
Q1f, Q1r, R1 = 1000, 800, 100
Q2, R2 = 1000, 1000

# --- 2. CASCADED MPC CONTROLLERS (The "Teacher") ---
def calculate_d_safe(v):
    return HWT * max(v, 0) + dss

def mpc1_high_level(v_ego, vf, vr, dr_f, dr_r, prev_vref, prev_ae):
    """ High-level MPC1: Computes optimal reference speed """
    def cost_func(ae):
        # Predict states
        vref_next = prev_vref + ae * Ts
        dr_f_next = dr_f + (vf - prev_vref)*Ts - 0.5*ae*(Ts**2)
        dr_r_next = dr_r + (prev_vref - vr)*Ts + 0.5*ae*(Ts**2)

        # Penalize errors from safe distances
        err_f = max(0, calculate_d_safe(vref_next) - dr_f_next)
        err_r = max(0, calculate_d_safe(vr) - dr_r_next)

        return (Q1f * err_f**2) + (Q1r * err_r**2) + R1 * ((ae - prev_ae)**2)

    res = minimize(cost_func, x0=prev_ae, bounds=[(a_min, a_max)])
    opt_ae = res.x[0]
    opt_vref = np.clip(prev_vref + opt_ae * Ts, 0, v_set)
    return opt_vref, opt_ae

def mpc2_low_level(v_ego, v_ref, prev_u):
    """ Low-level MPC2: Computes throttle/brake command [-1, 1] """
    def cost_func(u):
        v_next = v_ego + (u * kT / mass) * Ts
        return Q2 * ((v_next - v_ref)**2) + R2 * ((u - prev_u)**2)

    res = minimize(cost_func, x0=prev_u, bounds=[(-1.0, 1.0)])
    return res.x[0]

# --- 3. DATASET GENERATION LOOP ---
def generate_custom_dataset(num_samples=8000):
    print(f"Running simulation to generate {num_samples} records...")
    data = []

    # Initial State
    ve = 0.30
    vf = 0.35
    vr = 0.25
    df = 1.0
    dr = 1.5

    # Internal MPC memory states
    prev_vref = ve
    prev_ae = 0.0
    prev_u = 0.0

    for i in range(num_samples):
        # Create dynamic traffic scenarios (speeds change every 200 ticks)
        if i % 200 == 0:
            vf_target = np.clip(vf + random.uniform(-0.15, 0.15), 0, v_set)
            vr_target = np.clip(vr + random.uniform(-0.15, 0.15), 0, v_set)

        # Smooth transition of surrounding vehicle speeds
        vf += (vf_target - vf) * 0.05
        vr += (vr_target - vr) * 0.05

        # Calculate thresholds and errors
        d_safe_f = calculate_d_safe(ve)
        d_safe_r = calculate_d_safe(vr)
        delta_df = df - d_safe_f  # Error between actual and safe front distance

        # Run the MPCs
        vref_opt, ae_opt = mpc1_high_level(ve, vf, vr, df, dr, prev_vref, prev_ae)
        u_total = mpc2_low_level(ve, vref_opt, prev_u)

        # ----------------------------------------------------
        # LOG EXACTLY YOUR REQUESTED FEATURES AND LABELS
        # ----------------------------------------------------
        data.append({
            've': ve,
            'vf': vf,
            'vr': vr,
            'df': df,
            'dr': dr,
            'd_safe_f': d_safe_f,
            'd_safe_r': d_safe_r,
            'delta_df': delta_df,
            'u_total': u_total
        })

        # Step Plant Dynamics Forward
        ve += (u_total * kT / mass) * Ts
        ve = np.clip(ve, 0, v_set)
        df += (vf - ve) * Ts
        dr += (ve - vr) * Ts

        # Prevent physical crashes in the simulation loop
        df = max(0.01, df)
        dr = max(0.01, dr)

        # Update memory
        prev_vref = vref_opt
        prev_ae = ae_opt
        prev_u = u_total

    return pd.DataFrame(data)

# --- 4. EXECUTE & PREPARE FOR ML ---
# Generate the data
df_data = generate_custom_dataset(8000)

# Verify no NaN values
df_data.dropna(inplace=True)

# Define your requested features
features = [
    've',        # Ego velocity
    'vf',        # Front vehicle velocity
    'vr',        # Rear vehicle velocity
    'df',        # Distance to front vehicle
    'dr',        # Distance to rear vehicle
    'd_safe_f',  # Safe front distance threshold
    'd_safe_r',  # Safe rear distance threshold
    'delta_df'   # Error between actual and safe front distance
]

# Create X and y matrices
X = df_data[features]
y = df_data['u_total']

# Save to CSV for later use
df_data.to_csv('dual_aware_ml_dataset.csv', index=False)

print("\n--- Dataset Ready ---")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nSample Data (First 5 rows):")
print(X.head())

Running simulation to generate 8000 records...

--- Dataset Ready ---
X shape: (8000, 8)
y shape: (8000,)

Sample Data (First 5 rows):
    ve        vf        vr        df        dr  d_safe_f  d_safe_r  delta_df
0  0.3  0.343351  0.250407  1.000000  1.500000      0.46  0.425285  0.540000
1  0.3  0.337035  0.250793  1.004335  1.504959      0.46  0.425555  0.544335
2  0.3  0.331034  0.251160  1.008039  1.509880      0.46  0.425812  0.548039
3  0.3  0.325334  0.251508  1.011142  1.514764      0.46  0.426056  0.551142
4  0.3  0.319918  0.251840  1.013675  1.519613      0.46  0.426288  0.553675
